# 실험 2. 앙상블 모델 비교

실험 기록의 R3 특성 조합을 고정하고 LightGBM, XGBoost, CatBoost를 날짜 단위 검증으로 비교합니다. 각 모델은 MAE 목적함수를 사용합니다. CatBoost는 실행 시간이 길 수 있습니다.

검증은 학습 데이터 날짜를 하루씩 제외하는 방식으로 진행합니다. 무작위 분할은 사용하지 않습니다.

## 1. 환경 준비

Python 3.12 커널을 선택한 뒤 필요하면 아래 설치 셀을 한 번 실행하고 커널을 재시작합니다.

In [ ]:
import sys
assert (3, 12) <= sys.version_info[:2] <= (3, 14), f"Python 3.12 환경을 선택하세요. 현재: {sys.version.split()[0]}"
%pip install -r ../requirements.txt

## 2. 데이터 불러오기

처음에는 연습용 데이터(practice)로 실행하고, 대회 때는 PHASE를 inal로 변경합니다.

In [ ]:
from pathlib import Path
import pandas as pd

PHASE = 'practice'  # 'practice' 또는 'final'
REPO_ID = {'practice': 'CAMUS-LAB/bis-travel-time-practice',
           'final': 'CAMUS-LAB/bis-travel-time-final'}[PHASE]
DATA = Path('../data') / PHASE
if not (DATA / 'train.parquet').exists():
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=REPO_ID, repo_type='dataset', local_dir=DATA)

train = pd.read_parquet(DATA / 'train.parquet')
test = pd.read_parquet(DATA / 'test.parquet')
train['date'] = train['date'].astype(str)
print('train:', train.shape, '| test:', test.shape)
print('train dates:', sorted(train.date.unique()))
print('test labels available:', 'tt_sec' in test.columns)

## 3. R3 특성 준비 및 검증

각 날짜를 한 번씩 검증 데이터로 제외합니다. 점수는 모델 선택용이며, practice test 정답은 사용하지 않습니다.

In [ ]:
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

PEAK = {7, 8, 9, 17, 18, 19}
CAT = ['route_id', 'from_stop', 'to_stop']
NUM = ['route_type', 'direction', 'from_lat', 'from_lon', 'to_lat', 'to_lon',
       'dlat', 'dlon', 'stop_seq', 'dist_m', 'arrive_hour', 'is_weekend']

def add_features(df):
    out = df.copy()
    day = pd.to_datetime(out['date']).dt.dayofweek
    out['is_weekend'] = (day >= 5).astype('int8')
    out['dlat'] = out['to_lat'] - out['from_lat']
    out['dlon'] = out['to_lon'] - out['from_lon']
    return out

all_features = add_features(pd.concat([train, test], ignore_index=True))
CATEGORIES = {col: pd.Index(all_features[col].dropna().unique()) for col in CAT}

def prepare(df):
    x = add_features(df)[CAT + NUM].copy()
    for col in CAT:
        x[col] = pd.Categorical(x[col], categories=CATEGORIES[col])
    return x

def mae(y, pred):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(pred))))


def make_model(name):
    if name == 'LightGBM':
        return lgb.LGBMRegressor(
            objective='regression_l1', learning_rate=0.05, num_leaves=127,
            min_child_samples=10, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, n_estimators=1500, random_state=0, verbose=-1)
    if name == 'XGBoost':
        return xgb.XGBRegressor(
            objective='reg:absoluteerror', tree_method='hist', enable_categorical=True,
            learning_rate=0.05, max_depth=8, subsample=0.8, colsample_bytree=0.8,
            n_estimators=1200, early_stopping_rounds=100, random_state=0, n_jobs=-1)
    return CatBoostRegressor(
        loss_function='MAE', iterations=1200, depth=8, learning_rate=0.1,
        random_seed=0, verbose=False, allow_writing_files=False)


def fit_predict(name, fit, val):
    x_fit, x_val = prepare(fit), prepare(val)
    model = make_model(name)
    if name == 'LightGBM':
        model.fit(x_fit, fit['tt_sec'], eval_set=[(x_val, val['tt_sec'])],
                  eval_metric='l1', callbacks=[lgb.early_stopping(100, verbose=False)])
        pred = model.predict(x_val)
    elif name == 'XGBoost':
        model.fit(x_fit, fit['tt_sec'], eval_set=[(x_val, val['tt_sec'])], verbose=False)
        pred = model.predict(x_val)
    else:
        x_fit_cb, x_val_cb = x_fit.copy(), x_val.copy()
        for col in CAT:
            x_fit_cb[col] = x_fit_cb[col].astype(str)
            x_val_cb[col] = x_val_cb[col].astype(str)
        model.fit(x_fit_cb, fit['tt_sec'], cat_features=CAT,
                  eval_set=(x_val_cb, val['tt_sec']), verbose=False, early_stopping_rounds=100)
        pred = model.predict(x_val_cb)
    return pred

## 4. 모델별 4-fold 검증

XGBoost와 CatBoost는 환경에 따라 LightGBM보다 오래 걸릴 수 있습니다. 중단하더라도 셀 출력에 남은 검증 결과를 확인할 수 있습니다.

In [ ]:
rows = []
for model_name in ['LightGBM', 'XGBoost', 'CatBoost']:
    for val_date in sorted(train.date.unique()):
        fit = train[train.date != val_date]
        val = train[train.date == val_date]
        pred = fit_predict(model_name, fit, val)
        rows.append({'model': model_name, 'date': val_date,
                     'MAE': mae(val['tt_sec'], pred),
                     'weekday': pd.Timestamp(val_date).dayofweek < 5})
        print(f'{model_name} | {val_date} | MAE {rows[-1]["MAE"]:.3f}')

fold_scores = pd.DataFrame(rows)
summary_rows = []
for model_name, group in fold_scores.groupby('model'):
    summary_rows.append({'model': model_name,
                         '평일 MAE': group.loc[group['weekday'], 'MAE'].mean(),
                         '토요일 MAE': group.loc[~group['weekday'], 'MAE'].mean()})
summary = pd.DataFrame(summary_rows).set_index('model')
summary['4일 평균 MAE'] = (3 * summary['평일 MAE'] + summary['토요일 MAE']) / 4
summary.sort_values('4일 평균 MAE').round(3)